In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os, httpx
from dotenv import load_dotenv

# Load .env using absolute path (override=True to ensure it always loads fresh)
load_dotenv("/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/.env", override=True)

CA_BUNDLE = "/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/ca-bundle.pem"
os.environ["SSL_CERT_FILE"] = CA_BUNDLE
os.environ["REQUESTS_CA_BUNDLE"] = CA_BUNDLE

# Create a custom httpx client with the CA bundle
http_client = httpx.Client(verify=CA_BUNDLE)
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [15]:
from langchain_groq import ChatGroq

async_client = httpx.AsyncClient(verify=CA_BUNDLE)
# Using llama-3.3-70b-versatile for better tool call handling with nested objects
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    http_client=http_client,
    http_async_client=async_client,
)

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [18]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    llm,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="""You are a travel agent. No follow up questions.
When using search-flight:
- passengers must be an object like {"adults": 1, "children": 0, "infants": 0}, NOT a string
- dates must be in DD/MM/YYYY format
- for one-way flights, do NOT include returnDate at all
- flyFrom and flyTo should use IATA airport codes (e.g. SFO, NRT, HND)
"""
)

In [19]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st 2026")]},
    config
    )

In [20]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st 2026', additional_kwargs={}, response_metadata={}, id='f620215e-1f3d-440f-b71d-1908ff2befe1'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'nkpdz193z', 'function': {'arguments': '{"cabinClass":"M","curr":"USD","departureDate":"31/03/2026","flyFrom":"SFO","flyTo":"NRT","locale":"en","passengers":{"adults":1,"children":0,"infants":0}}', 'name': 'search-flight'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 2014, 'total_tokens': 2094, 'completion_time': 0.185257041, 'completion_tokens_details': None, 'prompt_time': 0.213622894, 'prompt_tokens_details': None, 'queue_time': 0.048960156, 'total_time': 0.398879935}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_ru

In [22]:
from IPython.display import Markdown, display

display(Markdown(response["messages"][-1].content))

Here are the results for a direct flight from San Francisco to Tokyo on March 31st, 2026:

| Airports | Departure and Arrival | Cabin Class | Price | Deep Link |
| --- | --- | --- | --- | --- |
| SFO → NRT | 31/03 19:20 → 01/04 06:25 (11h 5m) | Economy | $1324 | https://on.kiwi.com/oyrASI |
| SFO → NRT | 31/03 19:20 → 01/04 06:25 (11h 5m) | Economy | $1377 | https://on.kiwi.com/ZpI7Cd |
| SFO → NRT | 31/03 18:40 → 01/04 06:00 (11h 20m) | Economy | $1371 | https://on.kiwi.com/agvFPL |

The cheapest flights are:
- $521 for a flight with layovers: SFO → TPE → NRT, departing on 31/03 at 19:30 and arriving on 02/04 at 10:40, with a layover in Taipei (https://on.kiwi.com/fPAAoh)
- $521 for a flight with layovers: SFO → TPE → NRT, departing on 31/03 at 07:25 and arriving on 01/04 at 10:40, with a layover in Taipei (https://on.kiwi.com/2lvhU2)
- $615 for a flight with layovers: SFO → TPE → CTS → NRT, departing on 31/03 at 07:50 and arriving on 01/04 at 13:25, with layovers in Taipei and Sapporo (https://on.kiwi.com/0WBpWC)
- $627 for a flight with layovers: SFO → TPE → NRT, departing on 31/03 at 07:50 and arriving on 01/04 at 10:40, with a layover in Taipei (https://on.kiwi.com/10JfQY)
- $635 for a flight with layovers: SFO → YVR → ICN → NRT, departing on 31/03 at 17:15 and arriving on 02/04 at 00:45, with layovers in Vancouver and Seoul (https://on.kiwi.com/NT9PSX)
- $645 for a flight with layovers: SFO → TPE → NRT, departing on 31/03 at 07:50 and arriving on 01/04 at 10:40, with a layover in Taipei (https://on.kiwi.com/soR0hI)
- $766 for a flight with layovers: SFO → TPE → NRT, departing on 31/03 at 08:05 and arriving on 01/04 at 07:35, with a layover in Taipei (https://on.kiwi.com/hltvSQ)
- $812 for a flight with layovers: SFO → TPE → NRT, departing on 31/03 at 08:05 and arriving on 01/04 at 04:15, with a layover in Taipei (https://on.kiwi.com/ZdhbDo)
- $846 for a flight with layovers: SFO → TPE → NRT, departing on 31/03 at 07:25 and arriving on 01/04 at 03:25, with a layover in Taipei (https://on.kiwi.com/FdF5aH)
- $846 for a flight with layovers: SFO → TPE → NRT, departing on 31/03 at 07:50 and arriving on 01/04 at 03:25, with a layover in Taipei (https://on.kiwi.com/oQ9USY)
- $846 for a flight with layovers: SFO → TPE → NRT, departing on 31/03 at 07:50 and arriving on 01/04 at 04:25, with a layover in Taipei (https://on.kiwi.com/AWIDwj)

The shortest flights are:
- 11h 5m for a direct flight: SFO → NRT, departing on 31/03 at 19:20 and arriving on 01/04 at 06:25 (https://on.kiwi.com/oyrASI)
- 11h 5m for a direct flight: SFO → NRT, departing on 31/03 at 19:20 and arriving on 01/04 at 06:25 (https://on.kiwi.com/ZpI7Cd)
- 11h 20m for a direct flight: SFO → NRT, departing on 31/03 at 18:40 and arriving on 01/04 at 06:00 (https://on.kiwi.com/agvFPL)

Based on the results, I would recommend the flight with the layover in Taipei (https://on.kiwi.com/fPAAoh) as it is the cheapest option.

Have a nice trip to Tokyo! Did you know that Tokyo is home to the famous Tsukiji Fish Market, one of the largest fish markets in the world?